# Fetch DF and clean DF

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import OrdinalEncoder
df = pd.read_csv("data/anime_dataset.csv")

source_keep = {
    'Original', 'Manga', 'Light novel', 'Game', 'Visual novel',
    '4-koma manga', 'Novel', 'Web manga', 'Web novel'
}

type_keep = {
    'TV', 'Movie', 'OVA', 'TV Special', 'Special', 'ONA'
}

def clean_synopsis(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r"\(Source:.*?\)", "", text)
    text = re.sub(r"\[Written by.*?\]", "", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if "No synopsis information" in text:
        return ""

    return text

def split_pipe_field(value):
    if not isinstance(value, str) or value.strip() == "":
        return []
    return [item.strip().lower() for item in value.split("|") if item.strip()]

def clean_text_general(text):
    if pd.isna(text):
        return ""

    text = re.sub(r"\s+", " ", text)
    return text.strip()
    
def parse_duration(value):
    clean = value.lower().strip()
    clean = re.sub(r'per\s+ep\w*[\s\d]', '', clean)
    hours = re.search(r'(\d+)\s*hr', clean)
    minutes = re.search(r'(\d+)\s*min', clean)
    if hours:
        total += int(hours.group(1)) * 60
    if minutes:
        total += int(minutes.group(1))
    return total
def clean_rating(rating):
    if not isinstance(rating, str):
        return "unknown"

    rating = rating.strip()

    if rating.startswith("G"):
        return "all_ages"
    elif rating.startswith("PG -"):
        return "children"
    elif rating.startswith("PG-13"):
        return "teen"
    elif rating.startswith("R -"):
        return "mature"
    elif rating.startswith("R+"):
        return "mature_nudity"
    elif rating.startswith("Rx"):
        return "hentai"
    else:
        return "unknown"
    
def clean_df(df):
    numerics = [
    "score", "scored_by", "episodes", "rank", "popularity", "members", "favorites", "year"
    ]
    anime_df = df.copy()
    anime_df = anime_df.drop_duplicates(subset="mal_id").copy()
    anime_df = anime_df.dropna(subset=["score"]).copy()
    anime_df["display_title"] = anime_df["title_english"].fillna(anime_df["title"])
    anime_df["clean_synopsis"] = anime_df["synopsis"].apply(clean_synopsis)

    anime_df["genre_list"] = anime_df["genres"].apply(split_pipe_field)
    anime_df["themes_list"] = anime_df["themes"].apply(split_pipe_field)
    anime_df["demographics_list"] = anime_df["demographics"].apply(split_pipe_field)
    anime_df["studios_list"] = anime_df["studios"].apply(split_pipe_field)
    anime_df["producers_list"] = anime_df["producers"].apply(split_pipe_field)

    for cols in numerics:
        anime_df[cols] = pd.to_numeric(anime_df[cols],errors = "coerce") #Ensure that numeric cols are numeric Type: int, if cannot convert place as NaN

    anime_df = anime_df[anime_df["source"].isin(source_keep) & anime_df["type"].isin(type_keep)].copy()
    anime_df["type"] = anime_df["type"].apply(clean_text_general)
    anime_df["source"] = anime_df["source"].apply(clean_text_general)
    anime_df["status"] = anime_df["status"].apply(clean_text_general)
    anime_df["rating"] = anime_df["rating"].apply(clean_text_general)
    anime_df["rating_clean"] = anime_df["rating"].apply(clean_rating)
    

    anime_df["duration_normalized"] = anime_df.apply(parse_duration) #Duration field, might need for user preference
    anime_df["score_filled"] = anime_df["score"].fillna(anime_df["score"].mean()) #Replace NaN with the Mean of all scores

    anime_df["members_log"] = np.log1p(anime_df["members"].fillna(0)) #Compresses and normalizes large numbers
    anime_df["scored_by_log"] = np.log1p(anime_df["scored_by"].fillna(0))
    anime_df["favorites_log"] = np.log1p(anime_df["favorites"].fillna(0))

    anime_df["aired_from"] = pd.to_datetime(anime_df["aired_from"], errors="coerce")
    anime_df["year_clean"] = anime_df["year"].fillna(anime_df["aired_from"].dt.year)

df.columns
clean_df(df)

    








Index(['mal_id', 'title', 'title_english', 'title_japanese', 'type', 'source',
       'episodes', 'status', 'airing', 'aired_from', 'aired_to', 'duration',
       'rating', 'score', 'scored_by', 'rank', 'popularity', 'members',
       'favorites', 'season', 'year', 'studios', 'producers', 'licensors',
       'genres', 'themes', 'demographics', 'synopsis', 'image_url'],
      dtype='str')